# The Greeks Bearing Nets — Full Pipeline (Colab)

**Run this notebook on Google Colab with a T4 GPU runtime.**

Workflow:
1. Mount Google Drive for persistence
2. Clone repo + install deps
3. Upload manual data files (CBOE, Kaggle)
4. Run pipeline section by section — checkpoints save to Drive so you survive disconnects

**Runtime setup:** Runtime > Change runtime type > T4 GPU

---
## 0. Environment Setup

Run this section every time the runtime (re)starts.

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Project lives on Drive so it survives disconnects
import os
PROJECT_DIR = '/content/drive/MyDrive/greeks-bearing-nets'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'Project dir: {PROJECT_DIR}')

In [ ]:
# Clone repo (first time) or pull updates
REPO_URL = 'https://github.com/dma2459/greeks-bearing-nets.git'

REPO_DIR = '/content/greeks-bearing-nets'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# Install dependencies
!pip install -q yfinance fredapi tqdm scikit-learn

In [ ]:
# Symlink data and checkpoints to Drive (persist across disconnects)
import os

drive_dirs = {
    'data/raw':       f'{PROJECT_DIR}/data/raw',
    'data/processed': f'{PROJECT_DIR}/data/processed',
    'checkpoints':    f'{PROJECT_DIR}/checkpoints',
    'results':        f'{PROJECT_DIR}/results',
}

for local, remote in drive_dirs.items():
    os.makedirs(remote, exist_ok=True)
    # Ensure parent dir exists locally
    os.makedirs(os.path.dirname(local), exist_ok=True) if os.path.dirname(local) else None
    if os.path.islink(local):
        os.unlink(local)
    elif os.path.isdir(local):
        # Move contents to Drive, then remove local dir
        import shutil
        for item in os.listdir(local):
            s = os.path.join(local, item)
            d = os.path.join(remote, item)
            if not os.path.exists(d):
                shutil.move(s, d)
        shutil.rmtree(local)
    os.symlink(remote, local)
    print(f'  {local} -> {remote}')

print('\nDrive symlinks ready. Data persists across runtime restarts.')


In [ ]:
# Stale-cache detector — flags any preprocessed/raw file on Drive whose
# source code (in src/) has been touched more recently than the cache.
# Run this right after `git pull` so a source update doesn't silently sit
# behind a cached pickle. Examples that previously bit us: the SPY
# auto_adjust fix in download.py, the EOD-window shift in preprocess.py.
import os

_cache_rules = [
    ('data/raw/spy.csv',                    ['src/data/download.py']),
    ('data/raw/hyg.csv',                    ['src/data/download.py']),
    ('data/raw/lqd.csv',                    ['src/data/download.py']),
    ('data/processed/master_df.csv',        ['src/data/preprocess.py', 'src/data/download.py']),
    ('data/processed/scaler.pkl',           ['src/data/preprocess.py']),
    ('data/processed/train_sequences.npy',  ['src/data/preprocess.py']),
    ('data/processed/train_scaled.npy',     ['src/data/preprocess.py']),
    ('data/processed/test_scaled.npy',      ['src/data/preprocess.py']),
    ('data/processed/opts_train.pkl',       ['src/data/preprocess.py']),
    ('data/processed/opts_test.pkl',        ['src/data/preprocess.py']),
    ('data/processed/opts_sequences.npy',   ['src/data/preprocess.py']),
]

stale = []
for cache_rel, source_rels in _cache_rules:
    cache_path = os.path.join(REPO_DIR, cache_rel)
    if not os.path.exists(cache_path):
        continue
    cache_mtime = os.path.getmtime(cache_path)
    for src_rel in source_rels:
        src_path = os.path.join(REPO_DIR, src_rel)
        if os.path.exists(src_path) and os.path.getmtime(src_path) > cache_mtime:
            stale.append((cache_rel, src_rel))
            break

if stale:
    print('STALE CACHES — source updated after cache was written:')
    for c, s in stale:
        print(f'  {c}\n    <- newer: {s}')
    print('\nTo invalidate, run:')
    print('  for p in [')
    for c, _ in stale:
        print(f"      {os.path.join(REPO_DIR, c)!r},")
    print('  ]:')
    print('      if os.path.exists(p): os.remove(p); print(f"deleted {p}")')
else:
    print('All cached files are up to date with current source.')


In [ ]:
# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    device = torch.device('cuda')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')
    device = torch.device('cpu')

# Add repo to path
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

---
## 1. Data Download

Automated sources download here. Manual files must be uploaded to `data/raw/` on Drive.

In [ ]:
from src.data.download import download_all, download_yfinance, download_cboe_put_call_ratio, download_fred

raw_dir = os.path.join('data', 'raw')

# Check each data source independently
spy_exists = os.path.exists(os.path.join(raw_dir, 'spy.csv'))
pcr_exists = os.path.exists(os.path.join(raw_dir, 'put_call_ratio.csv'))
treasury_exists = os.path.exists(os.path.join(raw_dir, 'treasury.csv'))

if spy_exists and pcr_exists and treasury_exists:
    print('All market data already exists on Drive. Skipping download.')
else:
    if not spy_exists:
        print('Downloading yfinance data...')
        download_yfinance(data_dir=raw_dir)
    else:
        print('yfinance data exists. Skipping.')

    if not pcr_exists:
        print('Downloading CBOE put/call ratio...')
        download_cboe_put_call_ratio(data_dir=raw_dir)
    else:
        print('CBOE data exists. Skipping.')

    if not treasury_exists:
        print('Downloading treasury yields...')
        download_fred(data_dir=raw_dir)
    else:
        print('Treasury data exists. Skipping.')


In [ ]:
# FRED treasury yields are downloaded by download_all() above.
# If you need a FRED API key, set it in the download_all() call.
# As a fallback, you can also manually place treasury.csv in data/raw/.
treasury_path = os.path.join('data', 'raw', 'treasury.csv')
print(f'Treasury data exists: {os.path.exists(treasury_path)}')


In [ ]:
# The ONLY manual download needed: Kaggle SPY options data
# Everything else is automated now.
from src.data.download import load_options_data

raw_dir = os.path.join('data', 'raw')
try:
    _test_df = load_options_data(raw_dir)
    print(f'Options data found: {len(_test_df)} rows')
    del _test_df
except FileNotFoundError:
    print('OPTIONS DATA NOT FOUND.')
    print()
    print('Download from Kaggle and upload to Google Drive:')
    print('  https://www.kaggle.com/datasets/benjaminbtang/spy-options-2010-2023-eod')
    print(f'  Place CSV in: {PROJECT_DIR}/data/raw/')
    print()
    print('Or upload directly here (uncomment next lines):')
    # from google.colab import files
    # uploaded = files.upload()
    # for name, data in uploaded.items():
    #     with open(os.path.join(raw_dir, name), 'wb') as f:
    #         f.write(data)


In [ ]:
# Verify all data
import glob
files = sorted(glob.glob(os.path.join(raw_dir, '*.csv')))
print(f'Files in data/raw/ ({len(files)}):')
for f in files:
    size_mb = os.path.getsize(f) / 1e6
    print(f'  {os.path.basename(f):30s}  {size_mb:>8.2f} MB')

---
## 2. Preprocessing

In [ ]:
import numpy as np
import pandas as pd

from src.data.download import load_cboe_files
from src.data.preprocess import (
    build_master_dataframe, clean_and_split, fit_scaler,
    scale_data, build_sequences, preprocess_options, split_options,
    FEATURE_COLS, SEQ_LEN
)

RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'

# Check if already preprocessed
if os.path.exists(os.path.join(PROCESSED_DIR, 'train_sequences.npy')):
    print('Preprocessed data found on Drive. Loading...')
    master = pd.read_csv(os.path.join(PROCESSED_DIR, 'master_df.csv'), index_col=0, parse_dates=True)
    train_seqs = np.load(os.path.join(PROCESSED_DIR, 'train_sequences.npy'))
    print(f'Master: {master.shape}, Train sequences: {train_seqs.shape}')
    SKIP_PREPROCESS = True
else:
    SKIP_PREPROCESS = False
    print('No preprocessed data found. Running preprocessing...')

In [ ]:
if not SKIP_PREPROCESS:
    # Load raw data
    spy = pd.read_csv(os.path.join(RAW_DIR, 'spy.csv'), index_col=0, parse_dates=True)
    vix = pd.read_csv(os.path.join(RAW_DIR, 'vix.csv'), index_col=0, parse_dates=True)
    
    vvix_path = os.path.join(RAW_DIR, 'vvix.csv')
    vvix = pd.read_csv(vvix_path, index_col=0, parse_dates=True) if os.path.exists(vvix_path) else None
    
    hyg = pd.read_csv(os.path.join(RAW_DIR, 'hyg.csv'), index_col=0, parse_dates=True)
    lqd = pd.read_csv(os.path.join(RAW_DIR, 'lqd.csv'), index_col=0, parse_dates=True)
    dxy = pd.read_csv(os.path.join(RAW_DIR, 'dxy.csv'), index_col=0, parse_dates=True)
    
    treasury_path = os.path.join(RAW_DIR, 'treasury.csv')
    treasury = pd.read_csv(treasury_path, index_col=0, parse_dates=True) if os.path.exists(treasury_path) else None
    
    cboe = load_cboe_files(RAW_DIR)
    
    # Build master DataFrame
    master = build_master_dataframe(spy, vix, vvix, hyg, lqd, dxy, treasury, cboe)
    print(f'Master: {master.shape}, Date range: {master.index[0].date()} to {master.index[-1].date()}')
    
    # Clean, split, scale
    train_df, test_df = clean_and_split(master)
    print(f'Train: {len(train_df)} days, Test: {len(test_df)} days')
    
    scaler = fit_scaler(train_df, os.path.join(PROCESSED_DIR, 'scaler.pkl'))
    train_scaled = scale_data(train_df, scaler)
    test_scaled = scale_data(test_df, scaler)
    
    # Build sequences
    train_seqs = build_sequences(train_scaled, seq_len=SEQ_LEN)
    print(f'Training sequences: {train_seqs.shape}')
    
    # Save everything
    master.to_csv(os.path.join(PROCESSED_DIR, 'master_df.csv'))
    np.save(os.path.join(PROCESSED_DIR, 'train_sequences.npy'), train_seqs)
    np.save(os.path.join(PROCESSED_DIR, 'train_scaled.npy'), train_scaled)
    np.save(os.path.join(PROCESSED_DIR, 'test_scaled.npy'), test_scaled)
    train_df.index.to_frame().to_csv(os.path.join(PROCESSED_DIR, 'train_dates.csv'))
    test_df.index.to_frame().to_csv(os.path.join(PROCESSED_DIR, 'test_dates.csv'))
    print('Saved to Drive.')

In [ ]:
# Preprocess options (if file exists)
import pickle
from src.data.download import load_options_data

opts_test_path = os.path.join(PROCESSED_DIR, 'opts_test.pkl')
if os.path.exists(opts_test_path):
    print('Options data already preprocessed.')
else:
    try:
        opts_raw = load_options_data(RAW_DIR)
        print(f'Raw options: {len(opts_raw)} rows')

        with open(os.path.join(PROCESSED_DIR, 'scaler.pkl'), 'rb') as f:
            scaler = pickle.load(f)

        opts, opts_sequences = preprocess_options(opts_raw, master, scaler, seq_len=SEQ_LEN)
        opts_train, opts_test = split_options(opts)
        print(f'Train: {len(opts_train)}, Test: {len(opts_test)}')
        print(f'Unique-date sequences: {opts_sequences.shape}')

        opts_train.to_pickle(os.path.join(PROCESSED_DIR, 'opts_train.pkl'))
        opts_test.to_pickle(os.path.join(PROCESSED_DIR, 'opts_test.pkl'))
        np.save(os.path.join(PROCESSED_DIR, 'opts_sequences.npy'), opts_sequences)
        print('Saved to Drive.')
    except FileNotFoundError:
        print('No options CSV found. Upload to data/raw/ first.')


---
## 3. EDA (Quick Sanity Checks)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

master = pd.read_csv(os.path.join(PROCESSED_DIR, 'master_df.csv'), index_col=0, parse_dates=True)
FIGURES_DIR = 'results/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# SPY price
master['spy_close'].plot(ax=axes[0,0], title='SPY Close')
axes[0,0].axvline(pd.Timestamp('2020-01-01'), color='red', ls='--', label='Train/Test')
axes[0,0].legend()

# Volatility
for col in ['rv_21d', 'vix', 'gk_vol']:
    if col in master.columns:
        master[col].plot(ax=axes[0,1], alpha=0.7, label=col)
axes[0,1].set_title('Volatility Features')
axes[0,1].legend()

# Return distribution
returns = master['log_return'].dropna()
axes[1,0].hist(returns, bins=100, density=True, alpha=0.7)
axes[1,0].set_title(f'Log Returns (kurtosis={returns.kurtosis():.1f})')

# Regime distribution
regime_names = {0: 'Low', 1: 'Med', 2: 'High'}
for period, mask in [('Train', master.index < '2020'), ('Test', master.index >= '2020')]:
    counts = master.loc[mask, 'vol_regime'].value_counts().sort_index()
    axes[1,1].bar([f'{regime_names.get(int(k), k)}\n{period}' for k in counts.index],
                   counts.values, alpha=0.7, label=period)
axes[1,1].set_title('Vol Regime Counts')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_overview.png'), dpi=150, bbox_inches='tight')
plt.show()

# Correlation
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(master[FEATURE_COLS].corr(), annot=True, fmt='.1f', cmap='RdBu_r',
            center=0, ax=ax, annot_kws={'size': 7})
ax.set_title('Feature Correlations')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_correlation.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 4. TimeGAN Training

~30-60 min on T4 GPU. Checkpoints save to Drive every 100 epochs.

In [ ]:
from src.models.timegan import TimeGAN
from src.training.train_gan import train_timegan, generate_synthetic_sequences

train_seqs = np.load(os.path.join(PROCESSED_DIR, 'train_sequences.npy'))
GAN_CHECKPOINT = 'checkpoints/timegan'

# Check if already trained
gan_final_path = os.path.join(GAN_CHECKPOINT, 'timegan_final.pt')
if os.path.exists(gan_final_path):
    print('Trained GAN found on Drive. Loading...')
    model_gan = TimeGAN(input_dim=20, hidden_dim=64)
    model_gan.load_state_dict(torch.load(gan_final_path, map_location='cpu'))
    print('Loaded.')
    SKIP_GAN_TRAIN = True
else:
    SKIP_GAN_TRAIN = False
    print(f'No trained GAN found. Will train from scratch.')
    print(f'Training sequences: {train_seqs.shape}')

In [ ]:
if not SKIP_GAN_TRAIN:
    model_gan, history_gan = train_timegan(
        train_seqs,
        input_dim=20,
        hidden_dim=64,
        batch_size=128,
        phase1_epochs=200,
        phase2_epochs=200,
        phase3_epochs=300,
        lr=1e-3,
        device=device,
        checkpoint_dir=GAN_CHECKPOINT,
        log_every=20,
    )
    
    # Plot training curves
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(history_gan['phase1_loss']); axes[0].set_title('Phase 1: AE')
    axes[1].plot(history_gan['phase2_loss']); axes[1].set_title('Phase 2: Supervisor')
    axes[2].plot(history_gan['phase3_d_loss'], label='D')
    axes[2].plot(history_gan['phase3_g_loss'], label='G')
    axes[2].set_title('Phase 3: Adversarial'); axes[2].legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'gan_training_curves.png'), dpi=150)
    plt.show()

In [ ]:
# Generate 50K synthetic sequences
gan_seqs_path = os.path.join(PROCESSED_DIR, 'gan_sequences.npy')
if os.path.exists(gan_seqs_path):
    print('Synthetic sequences already on Drive.')
    gan_seqs = np.load(gan_seqs_path)
else:
    gan_seqs = generate_synthetic_sequences(
        model_gan, n_samples=50000, seq_len=60, batch_size=1000,
        device=device, save_path=gan_seqs_path,
    )
print(f'Synthetic sequences: {gan_seqs.shape}')

---
## 5. GAN Quality Checks

Must pass before proceeding.

In [ ]:
from src.evaluation.gan_eval import run_all_quality_checks

train_seqs = np.load(os.path.join(PROCESSED_DIR, 'train_sequences.npy'))
gan_seqs = np.load(os.path.join(PROCESSED_DIR, 'gan_sequences.npy'))

n_val = len(train_seqs) // 5
results_gan = run_all_quality_checks(
    train_seqs[:-n_val], train_seqs[-n_val:], gan_seqs,
    figures_dir=FIGURES_DIR,
)

print(f'\nKurtosis — Real: {results_gan["real_kurtosis"]:.2f}, Synthetic: {results_gan["fake_kurtosis"]:.2f}')
print(f'MMD — Baseline: {results_gan["baseline_mmd"]:.6f}, GAN: {results_gan["gan_mmd"]:.6f}')
print(f'MMD check: {"PASS" if results_gan["passed"] else "FAIL"}')

---
## 6. Generate Transformer Training Data

50K Heston + 50K GAN labeled samples.

In [ ]:
from src.models.heston import simulate_heston
from tqdm import tqdm

heston_labels_path = os.path.join(PROCESSED_DIR, 'heston_labels.npy')

if os.path.exists(heston_labels_path):
    print('Heston data already on Drive.')
    heston_contracts = np.load(os.path.join(PROCESSED_DIR, 'heston_contracts.npy'))
    heston_labels = np.load(heston_labels_path)
else:
    N_HESTON = 50000
    rng = np.random.RandomState(42)

    S0_samples = rng.uniform(250, 500, N_HESTON)
    moneyness_samples = rng.uniform(0.85, 1.15, N_HESTON)
    K_samples = S0_samples / moneyness_samples
    T_samples = rng.uniform(7/365, 90/365, N_HESTON)
    r_samples = rng.uniform(0.001, 0.055, N_HESTON)

    heston_labels = np.zeros(N_HESTON)
    heston_contracts = np.zeros((N_HESTON, 4))

    for i in tqdm(range(N_HESTON), desc='Heston MC'):
        price = simulate_heston(
            S0=S0_samples[i], K=K_samples[i], T=T_samples[i], r=r_samples[i],
            n_paths=10000, n_steps=max(1, int(T_samples[i] * 252)), seed=42+i,
        )
        heston_labels[i] = price
        heston_contracts[i] = [K_samples[i], T_samples[i], r_samples[i], moneyness_samples[i]]

    np.save(os.path.join(PROCESSED_DIR, 'heston_contracts.npy'), heston_contracts)
    np.save(heston_labels_path, heston_labels)
    print(f'Saved. Mean label: ${heston_labels.mean():.2f}')

In [ ]:
# GAN-based training data — risk-neutral corrected, vectorized.
#
# What changed from v1 (which produced gan_labels.npy):
#   1. Risk-neutral drift correction. The GAN learns physical-measure log
#      returns with ~7%/yr drift, but call pricing requires r-drift. v1
#      used the raw cumsum, which biased every label upward by mu-r.
#   2. 100 -> 2000 MC paths per contract (~4.5x lower label std-error).
#   3. Vectorized over CHUNK contracts at a time, ~50x faster despite 20x
#      more paths.
#
# Output renamed to gan_labels_rn.npy / gan_contracts_rn.npy so the v1
# cache can co-exist on Drive without shadowing the new labels.
gan_labels_path    = os.path.join(PROCESSED_DIR, 'gan_labels_rn.npy')
gan_contracts_path = os.path.join(PROCESSED_DIR, 'gan_contracts_rn.npy')

if os.path.exists(gan_labels_path) and os.path.exists(gan_contracts_path):
    print('GAN training data (rn) already on Drive.')
    gan_contracts = np.load(gan_contracts_path)
    gan_labels    = np.load(gan_labels_path)
else:
    gan_seqs = np.load(os.path.join(PROCESSED_DIR, 'gan_sequences.npy'))
    with open(os.path.join(PROCESSED_DIR, 'scaler.pkl'), 'rb') as f:
        scaler = pickle.load(f)

    log_return_mean = float(scaler.mean_[0])
    log_return_std  = float(scaler.scale_[0])

    # Un-scale once: (M, max_steps) physical-measure log returns.
    lr_seq = (gan_seqs[:, :, 0].astype(np.float32) * log_return_std
              + log_return_mean)
    M, max_steps = lr_seq.shape
    print(f'GAN sequences: {M} of length {max_steps}')

    # Estimate physical-measure moments per step from the full GAN pool.
    # Used to build the risk-neutral drift shift below.
    mu_phys_step = float(lr_seq.mean())
    sigma2_step  = float(lr_seq.var())
    print(f'GAN per-step  mean={mu_phys_step:.6f}  var={sigma2_step:.6f}'
          f'  -> annual drift={mu_phys_step*252:.3f}, vol={np.sqrt(sigma2_step*252):.3f}')

    N_GAN   = 50000
    N_PATHS = 2000
    rng_gan = np.random.RandomState(123)
    S0_gan        = rng_gan.uniform(250, 500, N_GAN).astype(np.float32)
    moneyness_gan = rng_gan.uniform(0.85, 1.15, N_GAN).astype(np.float32)
    K_gan         = (S0_gan / moneyness_gan).astype(np.float32)
    T_gan         = rng_gan.uniform(7/365, 90/365, N_GAN).astype(np.float32)
    r_gan         = rng_gan.uniform(0.001, 0.055, N_GAN).astype(np.float32)
    n_steps_arr   = np.clip((T_gan * 252).astype(int), 1, max_steps)

    gan_labels    = np.zeros(N_GAN, dtype=np.float32)
    gan_contracts = np.stack([K_gan, T_gan, r_gan, moneyness_gan], axis=1)

    CHUNK = 200          # ~200 x 2000 x 60 x 4B ~= 96 MB peak
    arange_steps = np.arange(max_steps)

    for cs in tqdm(range(0, N_GAN, CHUNK), desc='GAN pricing (RN, vec)'):
        ce = min(cs + CHUNK, N_GAN)
        nc = ce - cs

        # Sample N_PATHS GAN sequences per contract.
        idx   = rng_gan.randint(0, M, size=(nc, N_PATHS))
        paths = lr_seq[idx]                                    # (nc, N_PATHS, max_steps)

        # Mask: keep only the first n_steps_i log returns per contract.
        n_steps_chunk = n_steps_arr[cs:ce]                     # (nc,)
        mask   = (arange_steps[None, :] < n_steps_chunk[:, None]).astype(np.float32)
        sum_lr = (paths * mask[:, None, :]).sum(axis=2)        # (nc, N_PATHS)

        # Risk-neutral drift correction: shift each step's log return so
        # the path drift becomes (r - 0.5*sigma_annual^2)/252. Since
        # sigma2_step is the per-step variance (~ sigma_annual^2/252),
        # 0.5*sigma_annual^2/252 == 0.5*sigma2_step.
        r_chunk        = r_gan[cs:ce]
        shift_per_step = r_chunk / 252.0 - 0.5 * sigma2_step - mu_phys_step
        total_shift    = n_steps_chunk.astype(np.float32) * shift_per_step
        log_S_T_over_S0 = sum_lr + total_shift[:, None]        # (nc, N_PATHS)

        S_T     = S0_gan[cs:ce, None] * np.exp(log_S_T_over_S0)
        payoffs = np.maximum(S_T - K_gan[cs:ce, None], 0.0)
        prices  = np.exp(-r_chunk * T_gan[cs:ce]) * payoffs.mean(axis=1)
        gan_labels[cs:ce] = prices.astype(np.float32)

    np.save(gan_contracts_path, gan_contracts)
    np.save(gan_labels_path,    gan_labels)
    print(f'Saved {N_GAN} labels.  '
          f'mean ${gan_labels.mean():.2f}  median ${np.median(gan_labels):.2f}  '
          f'min ${gan_labels.min():.2f}  max ${gan_labels.max():.2f}')


---
## 7. Train Transformer (Experiments A, B, C)

~10-20 min each on T4.

In [ ]:
from src.models.transformer import build_transformer
from src.training.train_transformer import train_transformer
from src.data.dataset import SimulatedPricingDataset
from src.data.preprocess import prepare_pricing_sequences

# Raw real-market sequences (N, 60, 20). Keep around for ablations that
# need the full context or all features.
train_seqs_raw = np.load(os.path.join(PROCESSED_DIR, 'train_sequences.npy'))

h_contracts = np.load(os.path.join(PROCESSED_DIR, 'heston_contracts.npy'))
h_labels    = np.load(os.path.join(PROCESSED_DIR, 'heston_labels.npy'))
# v3: risk-neutral corrected GAN labels (cell 27).
g_contracts = np.load(os.path.join(PROCESSED_DIR, 'gan_contracts_rn.npy'))
g_labels    = np.load(os.path.join(PROCESSED_DIR, 'gan_labels_rn.npy'))

# Run 3 baseline input: drop 4 macro features + shorten context to 30 steps.
# These are the two ablation-validated changes (A2: -$0.30, A3: -$0.29).
train_seqs = prepare_pricing_sequences(train_seqs_raw)
print(f'Raw sequences: {train_seqs_raw.shape} -> Run 3 baseline: {train_seqs.shape}')

COMMON = dict(batch_size=256, max_epochs=100, lr=1e-4, patience=10, lr_patience=5, lr_factor=0.5, device=device)

histories = {}


In [ ]:
# Experiment A: Heston only — Run 4 (v2 checkpoints force retrain after
# contract-feature augmentation widened inputs from 20 → 24 channels).
ckpt_a = 'checkpoints/transformer/experiment_a_v2'
if os.path.exists(os.path.join(ckpt_a, 'best_model.pt')):
    print('Experiment A (v2) already trained.')
else:
    ds_a = SimulatedPricingDataset(train_seqs, h_contracts, h_labels)
    model_a, histories['A'] = train_transformer(
        build_transformer(), ds_a, checkpoint_dir=ckpt_a,
        experiment_name='Experiment A (Heston)', **COMMON)


In [ ]:
# Experiment B: GAN only — v3 (risk-neutral corrected GAN labels).
ckpt_b = 'checkpoints/transformer/experiment_b_v3'
if os.path.exists(os.path.join(ckpt_b, 'best_model.pt')):
    print('Experiment B (v3) already trained.')
else:
    ds_b = SimulatedPricingDataset(train_seqs, g_contracts, g_labels)
    model_b, histories['B'] = train_transformer(
        build_transformer(), ds_b, checkpoint_dir=ckpt_b,
        experiment_name='Experiment B (GAN, RN-corrected)', **COMMON)


In [ ]:
# Experiment C: Mixed Heston + GAN — v3 (risk-neutral GAN labels).
# Step 2 optimization: ATM + vol weighted loss focuses training on the
# regions where Run 3 trailed BS ($2.20 gap at ATM, $1.46 at high vol).
# atm_alpha=2.0 gives deep-ATM samples ~3x loss weight; vol_alpha=1.0
# adds up to ~2x boost for high vega-normalized time value.
ckpt_c = 'checkpoints/transformer/experiment_c_v3'
if os.path.exists(os.path.join(ckpt_c, 'best_model.pt')):
    print('Experiment C (v3) already trained.')
else:
    mixed_c = np.concatenate([h_contracts[:25000], g_contracts[:25000]])
    mixed_l = np.concatenate([h_labels[:25000],    g_labels[:25000]])
    ds_c = SimulatedPricingDataset(train_seqs, mixed_c, mixed_l)
    model_c, histories['C'] = train_transformer(
        build_transformer(), ds_c, checkpoint_dir=ckpt_c,
        experiment_name='Experiment C (Mixed RN-GAN, ATM+vol weighted)',
        loss_weighting='atm_vol', atm_alpha=2.0, vol_alpha=1.0,
        **COMMON)


In [ ]:
# Training curves
if histories:
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = {'A': 'blue', 'B': 'green', 'C': 'red'}
    for name, h in histories.items():
        ax.plot(h['val_loss'], color=colors[name], label=f'{name} (val)')
    ax.set_title('Transformer Validation Loss'); ax.set_xlabel('Epoch'); ax.set_ylabel('MSE')
    ax.legend(); ax.set_yscale('log')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'transformer_training.png'), dpi=150)
    plt.show()

---
## 8. Evaluation on Real Options (2020-2023)

In [ ]:
from src.evaluation.pricing_eval import evaluate_all_models
from src.data.preprocess import prepare_pricing_sequences

opts_test = pd.read_pickle(os.path.join(PROCESSED_DIR, 'opts_test.pkl'))
# Keep the raw (N, 60, 20) array for ablations; the baseline eval uses the
# Run 3+ prepped version (N, 30, 16).
opts_sequences_raw = np.load(os.path.join(PROCESSED_DIR, 'opts_sequences.npy'))
opts_sequences = prepare_pricing_sequences(opts_sequences_raw)
master_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'master_df.csv'), index_col=0, parse_dates=True)
print(f'Test contracts: {len(opts_test)}, Sequences raw: {opts_sequences_raw.shape} -> prepped: {opts_sequences.shape}')

# Load trained checkpoints. A is unchanged (Heston pipeline didn't move),
# so it stays v2. B and C use the v3 checkpoints retrained on RN-corrected
# GAN labels (cell 27).
models = {}
for name, d in [('Transformer A (Heston)',     'experiment_a_v2'),
                ('Transformer B (GAN, RN)',    'experiment_b_v3'),
                ('Transformer C (Mixed+ATM)',  'experiment_c_v3')]:
    p = os.path.join('checkpoints/transformer', d, 'best_model.pt')
    if os.path.exists(p):
        m = build_transformer()
        m.load_state_dict(torch.load(p, map_location='cpu'))
        m.target_mode = 'time_value'
        m.eval()
        models[name] = m
        print(f'  Loaded {name}')
    else:
        print(f'  MISSING: {p}')

models['Black-Scholes'] = 'BS'


In [ ]:
results = evaluate_all_models(
    opts_test, models, sequences=opts_sequences,
    master_df=master_df, device=device,
    figures_dir=FIGURES_DIR, tables_dir='results/tables',
)

print('\n' + '=' * 60)
print('FINAL RESULTS')
print('=' * 60)
print(results.round(4).to_string())


---
## 9. Ablation Studies

In [ ]:
from src.models.transformer import build_ablation_transformer
from src.evaluation.pricing_eval import (
    predict_transformer, compute_metrics, modify_sequences_for_ablation
)

y_true = opts_test['mid_price'].values

# Baseline: Experiment C (v2). Same Run 4 config as the main eval, so we reuse
# opts_sequences (already prepped in cell_34).
baseline = build_transformer()
baseline.load_state_dict(torch.load('checkpoints/transformer/experiment_c_v2/best_model.pt', map_location='cpu'))
baseline.target_mode = 'time_value'  # re-tag because load_state_dict does not persist attrs
baseline.eval()
baseline_preds = predict_transformer(baseline, opts_test, sequences=opts_sequences, device=device)
baseline_mae = compute_metrics(y_true, baseline_preds)['MAE']
print(f'Baseline MAE: {baseline_mae:.4f}')


In [ ]:
from src.data.preprocess import prepare_ablation_sequences

# train_seqs_raw is defined in cell_28 (the raw (N, 60, 20) array).
# Each ablation gets a different feature/length prep applied to it.
mixed_contracts = np.concatenate([h_contracts[:25000], g_contracts[:25000]])
mixed_labels = np.concatenate([h_labels[:25000], g_labels[:25000]])

ablation_models = {}
for abl_id in ['A1', 'A2', 'A3', 'A4', 'A5', 'A6']:
    ckpt = f'checkpoints/transformer/ablation_{abl_id.lower()}_v2'
    abl_seqs = prepare_ablation_sequences(train_seqs_raw, abl_id)
    seq_len = abl_seqs.shape[1]
    if os.path.exists(os.path.join(ckpt, 'best_model.pt')):
        print(f'{abl_id}: already trained ({abl_seqs.shape})')
        m = build_ablation_transformer(abl_id, seq_len=seq_len)
        m.load_state_dict(torch.load(os.path.join(ckpt, 'best_model.pt'), map_location='cpu'))
        m.target_mode = 'time_value'  # matches train_transformer default
        m.eval()
        ablation_models[abl_id] = m
    else:
        print(f'{abl_id}: training ({abl_seqs.shape})')
        ds = SimulatedPricingDataset(abl_seqs, mixed_contracts, mixed_labels)
        m = build_ablation_transformer(abl_id, seq_len=seq_len)
        m, _ = train_transformer(m, ds, checkpoint_dir=ckpt,
                                 experiment_name=f'Ablation {abl_id}',
                                 loss_weighting='atm_vol', atm_alpha=2.0, vol_alpha=1.0,
                                 **COMMON)
        ablation_models[abl_id] = m


In [ ]:
# Evaluate A-ablations. modify_sequences_for_ablation expects the raw
# (N, 60, 20) array and returns the per-ablation prep.
abl_rows = [{'Ablation': 'Baseline (C v2)', 'MAE': baseline_mae, 'Delta': 0.0}]

for abl_id, model in ablation_models.items():
    print(f'Evaluating {abl_id}...')
    abl_seqs = modify_sequences_for_ablation(opts_sequences_raw, abl_id)
    preds = predict_transformer(model, opts_test, sequences=abl_seqs, device=device)
    m = compute_metrics(y_true, preds)
    abl_rows.append({'Ablation': abl_id, 'MAE': m['MAE'], 'MSE': m['MSE'],
                     'MAPE': m['MAPE'], 'Delta': m['MAE'] - baseline_mae})
    print(f'  MAE: {m["MAE"]:.4f} (delta: {m["MAE"]-baseline_mae:+.4f})')

abl_df = pd.DataFrame(abl_rows)
abl_df.to_csv('results/tables/ablation_a_results.csv', index=False)
print('\n' + abl_df.to_string(index=False))


---
## 9. Step 3: Architecture Sweep (B-series ablations)

Each B-variant tweaks ONE architectural knob from the Run 4 baseline (Experiment C v2 with ATM+vol weighted loss):

| ID | Knob | Value |
| -- | ---- | ----- |
| B1 | depth (n_layers) | 5 (baseline 3) |
| B2 | embedding (d_model) | 128 (baseline 64) |
| B3 | attention heads | 8 (baseline 4) |
| B4 | dropout | 0.3 (baseline 0.1) |
| B5 | LR schedule | linear warmup + cosine decay (Vaswani 2017) |
| B6 | prediction head | deep MLP (4 hidden layers) |


In [ ]:
# Train Step 3 B-series ablations. All share the Run 4 baseline dataset
# (mixed Heston+GAN) and target_mode='time_value' with ATM+vol weighting —
# the only knob changing between variants is the one listed in the table.
from src.training.train_transformer import run_ablation

# Reuse the prepped baseline sequences (16 market features, seq_len=30).
b_train_seqs = train_seqs  # already (N, 30, 16) from cell_28

for abl_id in ['B1', 'B2', 'B3', 'B4', 'B5', 'B6']:
    ckpt = f'checkpoints/transformer/ablation_{abl_id.lower()}_v2'
    if os.path.exists(os.path.join(ckpt, 'best_model.pt')):
        print(f'{abl_id}: already trained')
        m = build_ablation_transformer(abl_id, seq_len=b_train_seqs.shape[1])
        m.load_state_dict(torch.load(os.path.join(ckpt, 'best_model.pt'), map_location='cpu'))
        m.target_mode = 'time_value'
        m.eval()
        ablation_models[abl_id] = m
    else:
        print(f'{abl_id}: training')
        ds = SimulatedPricingDataset(b_train_seqs, mixed_contracts, mixed_labels)
        # run_ablation handles B5 (sets lr_schedule='warmup_cosine' via kwargs.setdefault)
        m, _ = run_ablation(
            abl_id, ds, seq_len=b_train_seqs.shape[1],
            checkpoint_dir=ckpt,
            loss_weighting='atm_vol', atm_alpha=2.0, vol_alpha=1.0,
            **COMMON,
        )
        ablation_models[abl_id] = m


In [ ]:
# Evaluate B-series on real test options. B-ablations all use the same
# (N, 30, 16) sequence prep as the C baseline, so we reuse opts_sequences.
b_rows = [{'Ablation': 'Baseline (C v2)', 'MAE': baseline_mae, 'Delta': 0.0}]

for abl_id in ['B1', 'B2', 'B3', 'B4', 'B5', 'B6']:
    if abl_id not in ablation_models:
        continue
    print(f'Evaluating {abl_id}...')
    preds = predict_transformer(ablation_models[abl_id], opts_test,
                                sequences=opts_sequences, device=device)
    m = compute_metrics(y_true, preds)
    b_rows.append({'Ablation': abl_id, 'MAE': m['MAE'], 'MSE': m['MSE'],
                   'MAPE': m['MAPE'], 'Delta': m['MAE'] - baseline_mae})
    print(f'  MAE: {m["MAE"]:.4f} (delta: {m["MAE"]-baseline_mae:+.4f})')

b_df = pd.DataFrame(b_rows)
b_df.to_csv('results/tables/ablation_b_results.csv', index=False)
print('\n' + b_df.to_string(index=False))


---
## Summary

In [ ]:
print('Pipeline complete. All results saved to Google Drive at:')
print(f'  {PROJECT_DIR}/results/tables/')
print(f'  {PROJECT_DIR}/results/figures/')
print(f'  {PROJECT_DIR}/checkpoints/')
print()
print('Files for the report:')
for d in ['results/figures', 'results/tables']:
    if os.path.exists(d):
        for f in sorted(os.listdir(d)):
            print(f'  {d}/{f}')

In [ ]:
# Push results/figures and results/tables to GitHub so the latest run is
# visible without exporting from Drive. Requires a Personal Access Token
# stored in Colab secrets as `GH_TOKEN`:
#   Colab left sidebar -> key icon -> Add new secret
#   Name: GH_TOKEN, Value: a fine-grained PAT with Contents: Read+Write
#   on dma2459/greeks-bearing-nets, then toggle Notebook access on.
#
# `results/` is a symlink to Drive, and git refuses to add files "beyond
# a symbolic link". So we snapshot the Drive files into a temporarily-real
# `results/` dir inside the repo, push, then restore the symlink so future
# eval writes keep persisting to Drive.
import os, shutil, subprocess, datetime as dt
from google.colab import userdata

DRIVE_RESULTS = f'{PROJECT_DIR}/results'
REPO_RESULTS  = os.path.join(REPO_DIR, 'results')

# Snapshot what's on Drive RIGHT NOW (before we touch the symlink)
to_push = []
for sub in ('figures', 'tables'):
    src_dir = os.path.join(DRIVE_RESULTS, sub)
    if os.path.isdir(src_dir):
        for fn in sorted(os.listdir(src_dir)):
            full = os.path.join(src_dir, fn)
            if os.path.isfile(full):
                to_push.append((sub, fn, full))

if not to_push:
    print('No results files found to commit.')
else:
    # Break the symlink and copy files into a real dir inside the repo
    if os.path.islink(REPO_RESULTS):
        os.unlink(REPO_RESULTS)
    elif os.path.isdir(REPO_RESULTS):
        shutil.rmtree(REPO_RESULTS)

    rel_paths = []
    for sub, fn, full in to_push:
        dst_dir = os.path.join(REPO_RESULTS, sub)
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy2(full, os.path.join(dst_dir, fn))
        rel_paths.append(f'results/{sub}/{fn}')

    try:
        subprocess.run(['git', 'config', 'user.email', 'colab@noreply.local'], cwd=REPO_DIR, check=True)
        subprocess.run(['git', 'config', 'user.name',  'Colab Pipeline'],       cwd=REPO_DIR, check=True)
        subprocess.run(['git', 'add', '-f'] + rel_paths, cwd=REPO_DIR, check=True)

        has_diff = subprocess.run(['git', 'diff', '--cached', '--quiet'], cwd=REPO_DIR).returncode
        if has_diff == 0:
            print('No new result changes vs HEAD; nothing to commit.')
        else:
            msg = f'Update results from Colab run ({dt.datetime.now():%Y-%m-%d %H:%M})'
            subprocess.run(['git', 'commit', '-m', msg], cwd=REPO_DIR, check=True)

            try:
                TOKEN = userdata.get('GH_TOKEN')
            except Exception as e:
                TOKEN = None
                print(f'GH_TOKEN secret not accessible ({e}). Committed locally only.')

            if TOKEN:
                push_url = f'https://{TOKEN}@github.com/dma2459/greeks-bearing-nets.git'
                r = subprocess.run(['git', 'push', push_url, 'main'],
                                   cwd=REPO_DIR, capture_output=True, text=True)
                if r.returncode == 0:
                    print(f'Pushed {len(rel_paths)} result files to GitHub.')
                else:
                    print('Push failed:')
                    print(r.stderr)
    finally:
        # Restore the symlink so future eval cells write back to Drive
        if os.path.isdir(REPO_RESULTS) and not os.path.islink(REPO_RESULTS):
            shutil.rmtree(REPO_RESULTS)
        if not os.path.exists(REPO_RESULTS):
            os.symlink(DRIVE_RESULTS, REPO_RESULTS)
            print('Restored results/ symlink -> Drive.')
